# CUSTOMER CHURN PREDICTION

# 1. Import Libraries

In [ ]:
# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Explainability & Saving
import shap
import joblib

# 2. Load Dataset and Overview

In [ ]:
df=pd.read_csv("Telco-Customer-Churn.csv")

In [ ]:
df.shape

### Observation:
The dataset contains 7,043 customer records and 21 features.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Observations
- Tenure ranges from 0–72 months, with a median of 29 months.
- MonthlyCharges vary between 18.25 and 118.75.
- SeniorCitizen is a binary feature (0/1) and is better interpreted using frequency counts.
- No apparent anomalies are observed in the numerical features.

# 3. Data cleaning

In [ ]:
df.isnull().sum()  

In [ ]:
# Convert TotalCharges to numeric (blank spaces become NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Remove rows with missing TotalCharges
df = df.dropna(subset=['TotalCharges'])

df['TotalCharges'].isnull().sum()

### Observations
- The dataset contained missing values in the `TotalCharges` column due to blank spaces.
- These values were converted into NaN and removed.

In [ ]:
df.duplicated().sum() 

### Observation:
No duplicate records were found in the dataset.

In [ ]:
df.drop('customerID', axis=1, inplace=True)

The `customerID` column was removed because it is a unique identifier.

Strip values in cells

In [ ]:
for col in df.select_dtypes(include=['object', 'string']).columns: 
    df[col] = df[col].str.strip()

In [ ]:
for col in df.select_dtypes(include=['object', 'string']).columns:
    print(col)
    print(df[col].unique())
    print("-"*15)

###  Observation:
Unique values of categorical columns were examined to understand feature categories and identify potential inconsistencies before encoding.

## Check Outliers

In [ ]:
plt.figure(figsize=(6, 2))

sns.boxplot(x=df["MonthlyCharges"])

plt.title("Boxplot of Monthly Charges")
plt.xlabel("Monthly Charges")

plt.show()

### Observation: 
No significant extreme outliers are observed `MonthlyCharges`. 

# 4. EDA

### Business Objective:
Identify patterns and factors that influence customer churn in order to reduce customer loss.

In [ ]:
plt.figure(figsize=(6, 4))

ax = sns.countplot(x='Churn', data=df)

# Add count and percentage labels
total = len(df)

for p in ax.patches:
    count = int(p.get_height())
    percentage = 100 * count / total
    ax.annotate(f'{count}\n({percentage:.1f}%)',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)

plt.title("Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Count")

plt.show()

### Observation:
The dataset shows class imbalance, where non-churn customers significantly outnumber churn customers. 

In [ ]:
num_cols = ['tenure','MonthlyCharges','TotalCharges']

for col in num_cols:
    sns.histplot(df[col],kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

### Observation
- There are high number of both short and long tern customers
- Monthly charges are broadly distributed across different pricing plans, with no significant skewness or extreme outliers.
- `TotalCharges` is right-skewed, indicating that most customers have lower total charges.

Tenure vs Churn

In [ ]:
sns.boxplot(x='Churn', y='tenure', data=df)
plt.title("Tenure vs Churn")
plt.show()

### Observation:
Customers with lower tenure are more likely to churn.

MonthlyCharge vs Churn

In [ ]:
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title("Monthly Charges vs Churn")
plt.show()

###  Observation
Customers who churn generally have higher monthly charges than those who do not churn.

Contract Type vs Churn

In [ ]:
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title("Contract Type vs Churn")
plt.show()

### Observation
Month-to-month contracts show the highest churn ***
Long-term contracts reduce churn risk

Payment Method vs Churn

In [ ]:
sns.countplot(y='PaymentMethod',hue='Churn',data=df)
plt.title("Payment Method vs Churn")
plt.show()

### Observation
- Electronic check users churn more
- Payment convenience impacts retention

Correlation Analysis

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(df[num_cols].corr(),
            annot=True,
            cmap='coolwarm')

plt.title("Correlation Heatmap")

plt.show()

### Observation
Tenure strongly correlates with TotalCharges,
MonthlyCharges weakly correlates with tenure

Conducted comprehensive exploratory data analysis to identify key churn drivers including tenure, contract type, monthly charges, and payment methods. Discovered that short-tenure customers on month-to-month contracts with higher charges have significantly higher churn risk, guiding feature engineering and model selection.

Retention strategies should prioritize short-tenure customers.

# 5. Feature Engineering

### Binary Encoding 

In [ ]:
binary_cols = ['Partner', 'Dependents',
               'PhoneService',
               'PaperlessBilling',
               'Churn']

for col in binary_cols:
    df[col] = df[col].map({'Yes':1, 'No':0})

df.head()

### One-Hot Encoding

In [ ]:
# Select categorical columns
cat_cols = df.select_dtypes(
    include=['object', 'string']
).columns

# One-Hot Encoding
df = pd.get_dummies(df,
                    columns=cat_cols,
                    drop_first=True)

# Convert boolean columns to integers
bool_cols = df.select_dtypes(
    include='bool'
).columns

df[bool_cols] = df[bool_cols].astype(int)

df.head()

### Skewness Analysis

In [ ]:
num_cols = ['tenure',
            'MonthlyCharges',
            'TotalCharges']

print(df[num_cols].skew())

# Log transformation on right-skewed feature
df['TotalCharges'] = np.log1p(df['TotalCharges'])

df.head()

Tenure group creation

In [ ]:
df['tenure_group'] = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                             labels=['0-1yr','1-2yr','2-4yr','4+yr'],
                             include_lowest=True)  
sns.countplot(x='tenure_group', hue='Churn', data=df)

plt.show()

Customers in the first year of subscription are more likely to churn.

In [ ]:
df.drop('tenure_group', axis=1, inplace=True)

Categorical features were encoded, domain-driven features were created, and numerical features were prepared for machine learning modeling.

# 6. Modelling

Train-test split

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,          
    random_state=42,
    stratify=y
)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

The dataset was split into training and testing sets using an 80:20 ratio while preserving class distribution through stratification.

Feature scaling

In [ ]:
scaler = StandardScaler()

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

X_train.head() 
X_train.dtypes

## Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

# Evaluation


print("Accuracy:", accuracy_score(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

print("Confusion Matrix:\n",
      confusion_matrix(y_test, y_pred))

### Observation:
The Logistic Regression model achieved an accuracy of approximately 80%. The model demonstrated strong overall performance, while achieving moderate recall in identifying churn customers.

## Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,          
    class_weight='balanced',
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))

print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

### Observation
The Random Forest model achieved significantly higher recall, enabling better identification of churn customers. Although overall accuracy decreased slightly, the model demonstrated stronger effectiveness in detecting at-risk customers, which is valuable for customer retention strategies.

Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=10,
    random_state=42
)

dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))

print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall:", recall_score(y_test, y_pred_dt))
print("F1 Score:", f1_score(y_test, y_pred_dt))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))

### Observation:
The Decision Tree model achieved balanced classification performance but was outperformed by the tuned Random Forest model, particularly in identifying churn customers.

## XGBoost

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb))

print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))

print("Confusion Matrix:\n",
      confusion_matrix(y_test, y_pred_xgb))

## 7. Hyperparameter tuning (on RF and XGBoost)

## Randomized Search on Random Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 8, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced']
}

rf_random = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

rf_random.fit(X_train, y_train)

rf_random.best_params_

## Tuned Random Forest

In [ ]:
best_rf = rf_random.best_estimator_

y_pred_rf_tuned = best_rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf_tuned))

print("Precision:", precision_score(y_test, y_pred_rf_tuned))
print("Recall:", recall_score(y_test, y_pred_rf_tuned))
print("F1 Score:", f1_score(y_test, y_pred_rf_tuned))

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf_tuned))

## Randomized Search on XGBoost

In [ ]:
xgb_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_random = RandomizedSearchCV(
    estimator=XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    ),

    param_distributions=xgb_param_grid,

    n_iter=20,
    cv=5,
    scoring='recall',
    n_jobs=-1,
    verbose=1
)

xgb_random.fit(X_train, y_train)

xgb_random.best_params_

## Tuned XGBoost

In [ ]:
best_xgb = xgb_random.best_estimator_

y_pred_xgb_tuned = best_xgb.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_xgb_tuned))
print("Precision:", precision_score(y_test, y_pred_xgb_tuned))
print("Recall:", recall_score(y_test, y_pred_xgb_tuned))
print("F1 Score:", f1_score(y_test, y_pred_xgb_tuned))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb_tuned))

## Model Comparison

| **Metric**    | **Logistic Regression** | **Decision Tree** | **Random Forest** | **XGBoost** | **Tuned Random Forest** | **Tuned XGBoost** |
| ------------- | ----------------------: | ----------------: | ----------------: | ----------: | ----------------------: | ----------------: |
| **Accuracy**  |                   0.798 |             0.783 |             0.719 |       0.792 |                   0.707 |             0.793 |
| **Precision** |                   0.643 |             0.608 |             0.483 |       0.627 |                   0.471 |             0.632 |
| **Recall**    |                   0.540 |             0.511 |             0.802 |       0.543 |                  **0.812**|             0.532 |
| **F1 Score**  |                   0.587 |             0.555 |             0.603 |       0.582 |                   0.596 |             0.578 |


The performance of multiple machine learning models was compared using Accuracy, Precision, Recall and F1-Score metrics to identify the most effective model for customer churn prediction.

## Conclusion
Hyperparameter tuning was performed to optimize model performance using recall as the primary evaluation metric. Although the tuned Random Forest showed a slight decrease in accuracy and precision, it achieved the highest recall, making it the most suitable model for identifying customers at risk of churn.

SMOTE was also applied to address class imbalance during model development. However, tuned Random Forest was retained as the final model, as it provided the best balance between recall and overall performance.

# 8. Important Features

In [ ]:
feature_importance = pd.DataFrame({

    'Feature': X_train.columns,

    'Importance': best_rf.feature_importances_
})

# Sort values
feature_importance = feature_importance.sort_values(
    
    by='Importance',
    
    ascending=False
)

# Top 10 Features
print(feature_importance.head(10))

plt.figure(figsize=(10,6))

plt.barh(

    feature_importance['Feature'].head(10),

    feature_importance['Importance'].head(10)
)

plt.xlabel("Importance Score")

plt.ylabel("Features")

plt.title("Top 10 Important Features")

plt.gca().invert_yaxis()

plt.show()

### Observation
The Tuned Random Forest model identified tenure, contract type, internet service, payment method, and billing-related features as the most influential factors in predicting customer churn.

# 9. SHAP

## Global SHAP Explainability

In [ ]:
# Sample data for faster execution
X_sample = X_test.sample(300, random_state=42)

explainer = shap.Explainer(best_rf, X_train)

shap_values = explainer(X_sample)

shap.plots.bar(shap_values[:, :, 1], max_display=10)

### Observation:
SHAP analysis was performed to interpret the Tuned Random Forest model and understand the contribution of individual features toward customer churn prediction. Contract type, tenure, and internet service were the most influential features affecting churn predictions.

## Local SHAP Explainability

In [ ]:
# Predict probabilities
pred_probs = best_rf.predict_proba(X_sample)[:, 1]

# Customer index
customer_index = 5

print("Customer Index:", customer_index)

print("Churn Probability:",
      pred_probs[customer_index])

# Local Explanation
shap.plots.waterfall(
    shap_values[customer_index, :, 1]
)

This plot explains how individual features contributed to the churn prediction of a specific customer.

## Why Recall is prioritized ?
In churn prediction, a false negative (missing a churner) can result in permanent loss of that customer and associated revenue. A false positive only wastes a small retention cost. Therefore, Recall is the primary optimization metric.

# Final Outcome

A machine learning pipeline was successfully developed to predict customer churn using classification algorithms such as Logistic Regression, Random Forest, Decision Tree, and XGBoost. Among all models, Tuned Random Forest was selected as the final model due to its high recall, making it well suited for identifying customers at risk of churning.

# 10. Save Model

In [ ]:
joblib.dump(best_rf, "customer_churn_model.pkl")
joblib.dump(scaler, "scaler.pkl")